# UDACITY Designing Your First Workflow - Step Functions

## Step Functions & SageMaker

In the prior exercises, we've been working with many small services. This can be overwhelming for a data scientist that wants to establish a consistent methodology for handling data. Step Functions is an orchestration service that can allow us to utilize SageMaker in a methodical and consistent way. Step Functions also integrates with Lambda, which can allow us to potentially automate our entire machine learning pipeline end-to-end. Let's get a handle on what a 'step' in a step function looks like.

In this exercise, you will create a preprocessing step and a training step. Then you will create a step function to chain the two steps.

## Exercise: Grant Permissions and install packages.

Attach the IAMFullAccess and the StepFunctionsFullAccess polices to your SageMaker execution role.

In [1]:
%%bash
pip install stepfunctions

## Exercise: Fill out preprocessing step.

The 'step' interface is designed to be quite similar to the Preprocessing Job in lesson 2. The main difference between these is the ability of a 'step' to interface with other steps. Given the successful outcome of a single step, the next step specified in a workflow will automatically continue. In our case, a training step will launch given the successful outcome of a preprocessing step. The preprocessing step has been encoded for you. Upload the preprocessing code 'HelloBlazePreprocess.py' and the zipped dataset 'reviews_Musical_Instruments_5.json.zip' to s3, and fill out the constants in the code below. 

Code below is the preprocessing step. Fill in the constants in the code.

In [2]:
from sagemaker.core.helper.session_helper import get_execution_role

# Get role to easily find it in the IAM AWS console
role = get_execution_role()
print(role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
arn:aws:iam::864107284170:role/service-role/AmazonSageMaker-ExecutionRole-20260926T092167


In [5]:
# import stepfunctions
# print(stepfunctions.__version__) <- deprecated package

In [6]:
# # from sagemaker import get_execution_role
# from sagemaker.core.helper.session_helper import get_execution_role
# from sagemaker.sklearn.processing import SKLearnProcessor
# from sagemaker.processing import ProcessingInput, ProcessingOutput
# from stepfunctions.steps.sagemaker import ProcessingStep
# import sagemaker

# role = get_execution_role()

# PREPROCESSING_JOB_NAME = "l3e3-preprocessing"
# input_data = 's3://sagemaker-us-east-1-864107284170/lesson3/e3/reviews_Musical_Instruments_5.json.zip'
# input_preprocessing_code = 's3://sagemaker-us-east-1-864107284170/lesson3/e3/HelloBlazePreprocess.py'
# sess = sagemaker.Session()

# sklearn_processor = SKLearnProcessor(framework_version='0.20.0',
#                                      role=role,
#                                      instance_type='ml.m5.large',
#                                      instance_count=1)


# processed_data_train = "{}{}/{}".format("s3://", sess.default_bucket(), 'hello_blaze_train_scikit')
# processed_data_test = "{}{}/{}".format("s3://", sess.default_bucket(), 'hello_blaze_test_scikit')

# inputs=[ProcessingInput(source=input_data, destination='/opt/ml/processing/input', input_name = 'input-1'),  ProcessingInput(source=input_preprocessing_code , destination='/opt/ml/processing/input/code', input_name = 'code')]


# outputs=[ProcessingOutput(source='/opt/ml/processing/output/train', destination=processed_data_train, output_name = 'train_data'), ProcessingOutput(source='/opt/ml/processing/output/test', destination=processed_data_test, output_name = 'test_data')]


# processing_step = ProcessingStep(
#     "SageMaker pre-processing step 4",
#     processor=sklearn_processor,
#     job_name=PREPROCESSING_JOB_NAME,
#     inputs=inputs,
#     outputs=outputs,
#     container_entrypoint=["python3", "/opt/ml/processing/input/code/HelloBlazePreprocess.py"],
# )


In [7]:
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core import image_uris
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output,
)
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import Compute, OutputDataConfig, InputData
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep

role = get_execution_role()
region = "us-east-1"
bucket = "sagemaker-us-east-1-864107284170"
prefix = "lesson3/e3"

pipeline_session = PipelineSession()

# ---------- Preprocessing ----------

sklearn_processor = ScriptProcessor(
    image_uri=image_uris.retrieve(
        framework="sklearn", region=region, version="1.2-1",
        py_version="py3", instance_type="ml.m5.large",
    ),
    instance_type="ml.m5.large",
    instance_count=1,
    role=role,
    sagemaker_session=pipeline_session,
)

processor_args = sklearn_processor.run(
    code="HelloBlazePreprocess.py",
    inputs=[
        ProcessingInput(
            input_name="input-1",
            s3_input=ProcessingS3Input(
                s3_uri=f"s3://{bucket}/{prefix}/reviews_Musical_Instruments_5.json.zip",
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="train_data",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{bucket}/{prefix}/output/train",
                local_path="/opt/ml/processing/output/train",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="test_data",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{bucket}/{prefix}/output/test",
                local_path="/opt/ml/processing/output/test",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
)

step_process = ProcessingStep(
    name="PreprocessReviews",
    step_args=processor_args,
)

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:341: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## Exercise: Fill out Training Step

Upon the success of the preprocessing step, we wish to execute a training step. A training step is defined below. Fill the constants in the code.

In [21]:
# from stepfunctions.steps.sagemaker import TrainingStep
# import boto3

# WORKFLOW_OUTPUT = "CHANGE THIS"
# TRAINING_JOB_NAME = "CHANGE THIS"

# region_name = boto3.Session().region_name
# container = sagemaker.image_uris.retrieve(
#     region=region_name, framework="blazingtext", version="latest"
# )

# helloBlazeEstimator = sagemaker.estimator.Estimator(
#     container,
#     role=role,
#     instance_count=1,
#     instance_type='ml.m5.large',
#     volume_size=30,
#     max_run=360000,
#     input_mode="File",
#     output_path=WORKFLOW_OUTPUT,
#     sagemaker_session=sess,
# )

# helloBlazeEstimator.set_hyperparameters(mode='supervised')

# training_step = TrainingStep(
#     "SageMaker Training Step",
#     estimator=helloBlazeEstimator,
#     data={"train": sagemaker.TrainingInput(processed_data_train, content_type="text/plain"), "validation": sagemaker.TrainingInput(processed_data_test, content_type="text/plain")},
#     job_name=TRAINING_JOB_NAME,
#     wait_for_completion=True,
# )

In [8]:
# ---------- Training ----------

model_trainer = ModelTrainer(
    training_image=image_uris.retrieve(
        region=region, framework="blazingtext", version="latest",
    ),
    role=role,
    compute=Compute(instance_type="ml.m5.large", instance_count=1),
    output_data_config=OutputDataConfig(
        s3_output_path=f"s3://{bucket}/{prefix}/model-output",
    ),
    hyperparameters={"mode": "supervised"},
    sagemaker_session=pipeline_session,
)

train_args = model_trainer.train(
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=step_process.properties.ProcessingOutputConfig.Outputs[
                "train_data"
            ].S3Output.S3Uri,
        ),
        InputData(
            channel_name="validation",
            data_source=step_process.properties.ProcessingOutputConfig.Outputs[
                "test_data"
            ].S3Output.S3Uri,
        ),
    ],
)

step_train = TrainingStep(name="TrainBlazeText", step_args=train_args)

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:341: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## Exercise: Create Workflow & Execute It. 

To link the steps, you'll need to create a role that is capable of doing so. Go to IAM and create a Step Functions role, and attach the CloudWatchEventsFullAccess and SageMakerFullAccess policies. Once done, make use of the above steps to create a workflow. Quick debugging tip: jobs must have a unique name; you'll need to rename job names when debugging. Consider creating a method that will dynamically create unique job names! 

In [9]:
# from stepfunctions.steps import Chain
# from stepfunctions.workflow import Workflow

# workflow_role = "CHANGE THIS"

# workflow_graph = Chain([processing_step, training_step])
# workflow = Workflow(
#     name="CHANGE THIS",
#     definition=workflow_graph,
#     role=workflow_role,
# )

# workflow.create()

# execution = workflow.execute(
#     inputs={
#         "PreprocessingJobName": PREPROCESSING_JOB_NAME,  # Each pre processing job (SageMaker processing job) requires a unique name,
#         "TrainingJobName": TRAINING_JOB_NAME  # Each Sagemaker Training job requires a unique name,       
#     }
# )

# execution_output = execution.get_output(wait=True)

In [10]:
# ---------- Pipeline ----------

pipeline = Pipeline(
    name="l3e3-pipeline",
    steps=[step_process, step_train],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=role)
execution = pipeline.start()

You can track the outcome of this workflow through a custom UI that gets generated! Check it out!

In [12]:
# execution.render_progress()

In [14]:
execution.wait()

In [15]:
print(execution.describe())

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:864107284170:pipeline/l3e3-pipeline', 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:864107284170:pipeline/l3e3-pipeline/execution/jae2a0bpprrd', 'PipelineExecutionDisplayName': 'execution-1790945621741', 'PipelineExecutionStatus': 'Succeeded', 'PipelineExperimentConfig': {'ExperimentName': 'l3e3-pipeline', 'TrialName': 'jae2a0bpprrd'}, 'CreationTime': datetime.datetime(2026, 10, 2, 12, 53, 41, 679000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 10, 2, 13, 1, 28, 983000, tzinfo=tzlocal()), 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:864107284170:user-profile/d-p90rzbh4vwna/default-20260926T092167', 'UserProfileName': 'default-20260926T092167', 'DomainId': 'd-p90rzbh4vwna', 'IamIdentity': {'Arn': 'arn:aws:sts::864107284170:assumed-role/AmazonSageMaker-ExecutionRole-20260926T092167/SageMaker', 'PrincipalId': 'AROA4SMGOY3FKLLFKZJ6J:SageMaker'}}, 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagemaker:us

In [16]:
for step in execution.list_steps():
    print(step["StepName"], step["StepStatus"])

TrainBlazeText Succeeded
PreprocessReviews Succeeded
